In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# make  dataset for example
from sklearn.datasets import make_blobs
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

In [ ]:
print(f"X_train shape: {X_train.shape}and X_train[:10]: {X_train[:1]} ")
print(f"y_train shape: {y_train.shape} and y_train[:10]: {y_train[:10]} ")    

In [ ]:
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=20, cmap='viridis')
plt.title("Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### The MNIST database:
- (Modified National Institute of Standards and Technology database) is a large database of handwritten digits that is commonly used for training various image processing systems

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Activation    
# from tf.keras.losses import SparseCategoricalCrossEntropy



In [ ]:
y_train = y_train.reshape(-1, 1)

###  model compilation: In below Example: 
- here $\alpha$ = 0.001,learning Rate, is the fixed, base learning rate, but Adam dynamically computes a unique, changing step size for every single weight in your model based on that number.
### The model below is implemented with the softmax as an activation in the final Dense layer. The loss function is separately specified in the compile directive.
- The loss function is SparseCategoricalCrossentropy. In this model, the softmax takes place in the last layer. The loss function takes in the softmax output which is a vector of probabilities.
### Just In Below Cell: **Model 1**: **The Flaw in Model 1 (Softmax -> Loss):**

In [ ]:
# **Model 1 (Softmax --> Loss)** 
model = Sequential([
Dense(units=25,activation='relu'),
Dense(units=10,activation='relu'),
Dense(units=4,activation='softmax')
])

model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001)
    )

model.fit(X_train,y_train,epochs=10)

### Below cell shows :
- A "proof of concept" test. It is trying to show you exactly that we have mentioned in the conclusion below:
- This below code on the model (which uses the Softmax activation in its final layer), the output serves as the final proof of why we call this the "non-preferred" approach for training.

    ### Here is the conclusion of what this code demonstrates:

    - The Outputs are Forced Probabilities: Unlike the raw logits from the preferred model, the p_nonpreferred[:2] printout will show neatly formatted decimals. 
    - Every number will be between 0.0 and 1.0, and the values in each list will sum to exactly 1.0.
- The Extremes are Bounded: The np.max will never exceed 1.0, and the np.min will never drop below 0.0.

In [ ]:

p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

### Here is summary of the Softmax loss and cost functions with proper mathematical equations:

---

### **1. Cross-Entropy Loss (Single Example)**
The **Loss** function evaluates the error for a *single* training example. For Softmax, we use the Cross-Entropy loss:

$$L(\mathbf{a}, y) = \begin{cases} -\log(a_1), & \text{if } y=1 \\ \vdots & \vdots \\ -\log(a_N), & \text{if } y=N \end{cases}$$

* $\mathbf{a}$: The output vector of the softmax function (a list of probabilities that sum to 1).
* $y$: The true target category for this example.
* **Key concept:** Only the probability predicted for the *correct* target class contributes to the loss. All other lines effectively become zero.

### **2. The Indicator Function**

To scale this logic up to a mathematical formula that handles all examples, we use an **indicator function**. It acts as a switch, turning on ($1$) only when looking at the correct target class and turning off ($0$) otherwise:

$$1\{y == n\} = \begin{cases} 1, & \text{if } y == n \\ 0, & \text{otherwise} \end{cases}$$

### **3. Total Cost Function (All Examples)**

While *Loss* is for one example, **Cost** ($J$) is the average of all the individual losses across your entire dataset. Combining the indicator function with the expanded Softmax formula gives us the final Cost equation:

$$J(\mathbf{w}, b) = -\frac{1}{m} \left[ \sum_{i=1}^{m} \sum_{j=1}^{N} 1\{y^{(i)} == j\} \log \left( \frac{e^{z_j^{(i)}}}{\sum_{k=1}^{N} e^{z_k^{(i)}}} \right) \right]$$

**Variables:**
* $m$: Total number of training examples.
* $N$: Total number of output categories (classes).
* $i$: The current example being evaluated.
* $j$: The current class being evaluated.
* $z$: The raw logit score before applying softmax. The fraction inside the logarithm is simply the Softmax probability $a_j^{(i)}$.

---
### Below we will see the prefered Model (i.e. Model 2) for the same problem and it is best prefered :

### Preferred Model (Softmax + Loss)

- More stable and accurate results can be obtained if the softmax and loss are combined during training.   
- This is enabled by the 'preferred' organization shown here.


In [ ]:
preferred_Model = Sequential([
    Dense(units=25,activation ='relu'),
    Dense(units =10,activation = 'relu'),
    Dense(units=4,activation='linear' )

])
preferred_Model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), # here we use from_logits = True because the output layer is linear and not softmax   
    optimizer=tf.keras.optimizers.Adam(0.001)
)
preferred_Model.fit(X_train,y_train,epochs=10)    # training the preferred model
# logits=preferred_Model(X_train)  # prediction from the preferred model (logits), here i will get z1,z2,... zn Not a1,a2,... an because the output layer is linear and not softmax
# f_x  = tf.nn.softmax(logits)  # applying softmax to the logits to get probabilities
# print(f"Logits: {logits[:2]}")  # print the first two logits
# print(f"Softmax probabilities: {f_x[:2]}")  # print the first two softmax probabilities



#### Output Handling
Notice that in the preferred model, the outputs are not probabilities, but can range from large negative numbers to large positive numbers. The output must be sent through a softmax when performing a prediction that expects a probability. 
Let's look at the preferred model outputs:

In [ ]:
p_preferred = preferred_Model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

### we can see the output is not Probabilities.
- if desired prediction is probabilities we  should use softmax to the output of the preferred model to get probabilities.


In [ ]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

In [ ]:
for i in range(6):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

## SparseCategorialCrossentropy or CategoricalCrossEntropy
Tensorflow has two potential formats for target values and the selection of the loss defines which is expected.
- SparseCategorialCrossentropy: expects the target to be an integer corresponding to the index. For example, if there are 10 potential target values, y would be between 0 and 9. 
- CategoricalCrossEntropy: Expects the target value of an example to be one-hot encoded where the value at the target index is 1 while the other N-1 entries are zero. An example with 10 potential target values, where the target is 2 would be [0,0,1,0,0,0,0,0,0,0].


 ### Learned the preferred model construction in Tensorflow:
    - No activation on the final layer (same as linear activation)
    - SparseCategoricalCrossentropy loss function
    - use from_logits=True
- Recognized that unlike ReLU and Sigmoid, the softmax spans multiple outputs.

### **The conclusion**: 
### **Logits vs. Softmax: Why from_logits=True is the Better Approach in Keras (Stop Using Softmax in Your Final Layer: Understanding from_logits=True)**
This is an excellent question and a very common source of confusion when building classification models in TensorFlow and Keras. 

The short answer is: **The first code snippet (`preferred_model` with `from_logits=True`) is the better, officially recommended approach.** 

Here is a detailed breakdown of exactly why this is the case, the underlying mechanics, and the one minor trade-off you need to keep in mind.

---

### **1. The Core Difference: Logits vs. Probabilities**

To understand the difference, we have to look at what the final layer is outputting:

* **Model 2 (`preferred_model`):** Uses a `linear` activation function. This means the model outputs **logits**—the raw, unscaled mathematical scores from the final neurons. These numbers can be anything (e.g., `-4.5`, `0.0`, `12.8`).
* **Model 1 (`model`):** Uses a `softmax` activation function. This forces the model to immediately squash those raw scores into **probabilities** (numbers between $0$ and $1$ that add up to $1.0$).

### **2. Why `from_logits=True` is Better (Numerical Stability)**

The reason the first approach is superior comes down to how computers handle extremely small numbers, a concept known as **numerical stability**. 

During training, the `SparseCategoricalCrossentropy` loss function calculates how wrong your model is by taking the natural logarithm of the predicted probability. 

* **The Flaw in Model 1 (Softmax -> Loss):**
    When you calculate the softmax first, a very wrong prediction might result in an incredibly tiny probability, like $0.000000000000000001$. Due to the memory limits of floating-point numbers in computers, Keras might accidentally round this tiny number down to exactly $0.0$. 
    When the loss function then tries to calculate the logarithm of zero ($\log(0)$), the math breaks. It results in negative infinity, which causes your gradients to explode and your model's loss to output `NaN` (Not a Number), crashing your training.
* **The Fix in Model 2 (from_logits=True):**
    When you set `from_logits=True`, you are telling Keras: *"Don't calculate the softmax yet. Take these raw numbers and do the Softmax and Cross-Entropy math together in one single step."* By combining these two formulas under the hood, TensorFlow can use an optimized mathematical shortcut (called the *LogSumExp* trick) that mathematically cancels out the extremes. It completely avoids the rounding errors, ensuring your model never tries to calculate $\log(0)$.

### **3. Summary Comparison**

| Feature | Model 2 (`preferred_model`) | Model 1 (`model`) |
| :--- | :--- | :--- |
| **Final Layer Activation** | `linear` (Raw Logits) | `softmax` (Probabilities) |
| **Loss Function Flag** | `from_logits=True` | `from_logits=False` (Default) |
| **Training Stability** | **Highly Stable** (Prevents `NaN` errors) | **Unstable** (Prone to floating-point errors) |
| **Computation Speed** | Slightly Faster (Combined math steps) | Slightly Slower (Separate math steps) |

### **The One Catch: Making Predictions Later**

Because `preferred_model` outputs raw logits, you have to remember that when training is finished and you want to use the model to make a prediction on new data, the output will just be raw numbers, not percentages. 

To get human-readable probabilities during inference, you just need to pass the output through a softmax function manually:

```python
# Getting raw logits
raw_predictions = preferred_model.predict(X_test)

# Converting them to readable probabilities
probabilities = tf.nn.softmax(raw_predictions)
```

---


# Library for the Mathmatical equation, differentiation and symbol and expression representation as we wrote on paper.

In [ ]:
import sympy # from sympy import symbols,diff,exp 
# library for symbolic mathematics, which can be used to compute derivatives and understand the softmax function better.

In [ ]:
j,w = sympy.symbols('j,w') # j is the loss function, w is the weight and b is the bias
j = w**-2 #w**6 #j = w**2
# j,w,b = sympy.symbols('j,w,b') # j is the loss function, w is the weight and b is the bias
# j = w**-2 + b #w**6 #j = w**2
j

In [ ]:
# j = w**-2 #w**6 #j = w**2
j = sympy.exp(w)/ (sympy.exp(w) + sympy.exp(w+1) + sympy.exp(w+2)) # example of softmax function for a single class (the first class) in a 3-class problem, where the logits are w, w+1, and w+2
# print(f"j: {j}")
j

In [ ]:
dj_dw = sympy.diff(j,w)
dj_dw

In [ ]:
dj_dw.subs([(w,2)]) # j differentiation value at w=2
# dj_dw